In [ ]:
import asyncio
import json
import random
import shutil
from bisect import bisect_right
from datetime import datetime, timezone
from math import gcd
from pathlib import Path
from threading import Event, Thread
from uuid import uuid4

import ipywidgets as widgets
import numpy as np
import sounddevice as sd
import soundfile as sf
from audiomentations import AddBackgroundNoise, AddGaussianSNR, Gain, TimeStretch
from IPython.display import SVG, Audio, display
from scipy.signal import resample_poly

previous_recorder = globals().get("_recorder") or globals().get("_noise_recorder")
if previous_recorder is not None and previous_recorder.running:
    raise RuntimeError("Stop recording before rerunning setup or the standalone cell.")
previous_status_task = globals().get("_recording_status_task")
if previous_status_task is not None:
    previous_status_task.cancel()

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the GPTWake project.")

DATA_DIR = ROOT / "data"
SAMPLE_RATE = 16000
SPEAKER_ID = "self"
state = {"audio": None, "source": None, "updating_picker": False}
print("Dataset root:", DATA_DIR)
print(
    "Layout: positive, negative, and hard_negative have raw/processed/augmented; noise uses raw only."
)
print("No folders or recordings are created until you save data.")


def log(*values, target=None):
    if target is None:
        target = globals().get("output")
    if target is None:
        print(*values)
    else:
        target.append_stdout(" ".join(str(value) for value in values) + "\n")


def emit_preview(value, target=None):
    # Update widget state directly; do not also publish a captured IOPub display.
    if target is None:
        target = globals().get("output")
    if target is None:
        display(value)
    else:
        target.append_display_data(value)


def close_panel(widget):
    for child in getattr(widget, "children", ()):
        close_panel(child)
    if hasattr(widget, "disabled"):
        widget.disabled = True
    widget.close()


def dataset_path(kind, stage):
    if kind not in {"positive", "negative", "hard_negative", "noise"} or stage not in {
        "raw",
        "processed",
        "augmented",
        "synthetic",
    }:
        raise ValueError("Unknown dataset category or stage.")
    if stage == "synthetic" and kind not in {"positive", "negative"}:
        raise ValueError("Synthetic inputs are supported for positive and negative categories.")
    return DATA_DIR / kind / stage


def next_path(directory, prefix, suffix=".wav"):
    # Inspect existing names on disk so restarting the kernel cannot overwrite a recording.
    number = 1
    for path in directory.glob(f"{prefix}_*"):
        tail = path.stem.rsplit("_", 1)[-1]
        if tail.isdigit():
            number = max(number, int(tail) + 1)
    return directory / f"{prefix}_{number:04d}{suffix}"


def write_metadata(path, metadata):
    path.write_text(
        json.dumps(
            metadata, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else str(x)
        ),
        encoding="utf-8",
    )


def write_wav(path, audio, sample_rate=SAMPLE_RATE, subtype="PCM_16", overwrite=True):
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim not in {1, 2} or not len(audio) or not np.isfinite(audio).all():
        raise ValueError("Expected nonempty, finite audio.")
    if sample_rate <= 0:
        raise ValueError("Sample rate must be positive.")
    peak = float(np.max(np.abs(audio)))
    if subtype != "FLOAT" and peak > 1:
        log(f"Reducing peak {peak:.3f} to avoid PCM clipping: {path.name}")
        audio = audio * (0.999 / peak)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb" if overwrite else "xb") as stream:
        sf.write(stream, audio, sample_rate, format="WAV", subtype=subtype)
    return audio


def normalize_source(path):
    audio, rate = sf.read(path, dtype="float32", always_2d=True)
    audio = audio.mean(axis=1)
    if rate != SAMPLE_RATE:
        divisor = gcd(rate, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // divisor, rate // divisor)
    return np.asarray(audio, dtype=np.float32)


def save_recording(kind, audio, sample_rate, metadata=None):
    path = next_path(dataset_path(kind, "raw"), kind)
    write_wav(path, audio, sample_rate, subtype="FLOAT", overwrite=False)
    write_metadata(
        path.with_suffix(".json"),
        {
            **(metadata or {}),
            "recording_id": path.stem,
            "category": kind,
            "label": {"positive": 1, "negative": 0, "hard_negative": 0, "noise": None}[kind],
            "speaker_id": SPEAKER_ID,
            "split": "unassigned",
            "raw_sample_rate": sample_rate,
            "created_at": datetime.now(timezone.utc).isoformat(),
        },
    )
    return path


class PositiveWindowError(ValueError):
    """The selected positive phrase cannot fit the requested output window."""


def positive_phrase(source, audio, bounds=None):
    method = "manual" if bounds is not None else "full_recording"
    if bounds is None:
        sidecar = dataset_path("positive", "processed") / f"{source.stem}.json"
        saved = json.loads(sidecar.read_text()) if sidecar.exists() else {}
        # Only explicit manual bounds are reused; legacy heuristic bounds are ignored.
        if saved.get("selection_method") == "manual":
            bounds = saved["selection_seconds"]
            method = "manual"
        else:
            bounds = (0.0, len(audio) / SAMPLE_RATE)
    start, end = map(float, bounds)
    if not 0 <= start < end <= len(audio) / SAMPLE_RATE:
        raise ValueError("Phrase bounds must lie within the original recording.")
    segment = audio[round(start * SAMPLE_RATE) : round(end * SAMPLE_RATE)]
    if not len(segment):
        raise ValueError("Select a nonempty phrase interval.")
    return segment, [start, end], method


def process_recording(kind, source, bounds=None, window_seconds=2.0):
    source = Path(source)
    if kind == "noise":
        return [source]
    sidecar = source.with_suffix(".json")
    source_metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
    if source_metadata.get("status", "complete") != "complete":
        raise ValueError("Only completed recordings can be processed.")
    if kind in {"negative", "hard_negative"}:
        raise ValueError(
            "Use the negative extraction panel to choose approved time intervals first."
        )
    audio = normalize_source(source)
    common = {
        "source": str(source.relative_to(ROOT)),
        "recording_id": source.stem,
        "category": kind,
        "label": {"positive": 1, "negative": 0, "hard_negative": 0, "noise": None}[kind],
        "sample_rate": SAMPLE_RATE,
        "speaker_id": source_metadata.get("speaker_id", SPEAKER_ID),
        "split": source_metadata.get("split", "unassigned"),
    }
    directory = dataset_path(kind, "processed")
    if kind == "positive":
        segment, interval, method = positive_phrase(source, audio, bounds)
        if not np.isfinite(window_seconds) or window_seconds <= 0:
            raise ValueError("Output window must be a positive finite duration.")
        window_size = round(window_seconds * SAMPLE_RATE)
        if len(segment) > window_size:
            raise PositiveWindowError(
                f"Phrase ({len(segment) / SAMPLE_RATE:.3f}s) exceeds the window "
                f"({window_seconds:.3f}s). Adjust Phrase or increase Window; "
                "processed audio was not saved or overwritten."
            )
        common["selection_method"] = method
        segments = [(source.stem, segment, interval)]
    outputs = []
    for stem, segment, interval in segments:
        path = directory / f"{stem}.wav"
        write_wav(path, segment)
        write_metadata(path.with_suffix(".json"), {**common, "selection_seconds": interval})
        outputs.append(path)
    return outputs


def read_normalized_segment(source, start_frame, frame_count):
    with sf.SoundFile(source) as stream:
        rate = stream.samplerate
        stream.seek(start_frame)
        audio = stream.read(frame_count, dtype="float32", always_2d=True)
    if len(audio) != frame_count or not len(audio) or not np.isfinite(audio).all():
        raise ValueError("The selected segment is empty, incomplete, or contains invalid samples.")
    audio = audio.mean(axis=1)
    if rate != SAMPLE_RATE:
        divisor = gcd(rate, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // divisor, rate // divisor)
    expected = round(frame_count * SAMPLE_RATE / rate)
    return np.asarray(audio[:expected], dtype=np.float32)


def extract_negative_windows(
    kind,
    source,
    intervals,
    window_seconds=2.0,
    mode="random",
    count=20,
    stride_seconds=None,
    split=None,
    seed=None,
):
    if kind not in {"negative", "hard_negative"}:
        raise ValueError("Choose negative or hard_negative for window extraction.")
    source = Path(source)
    if source.resolve().parent != dataset_path(kind, "raw").resolve():
        raise ValueError("The selected source must belong to this category's raw directory.")
    sidecar = source.with_suffix(".json")
    metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
    if metadata.get("status", "complete") != "complete":
        raise ValueError("Only completed recordings can be processed.")
    info = sf.info(source)
    if not np.isfinite(window_seconds) or window_seconds <= 0:
        raise ValueError("Window duration must be positive and finite.")
    window_size = round(window_seconds * SAMPLE_RATE)
    native_size = round(window_size * info.samplerate / SAMPLE_RATE)
    if window_size < 1 or native_size < 1 or info.frames < 1:
        raise ValueError("Source and window must contain audio samples.")
    if mode not in {"random", "sliding", "clips", "placement"}:
        raise ValueError("Unknown window extraction mode.")
    if mode in {"clips", "placement"} and kind != "hard_negative":
        raise ValueError("Clip padding is intended for separately recorded hard negatives.")
    if not intervals:
        raise ValueError("Enter at least one approved start,end interval in seconds.")
    ranges = []
    for interval in intervals:
        if len(interval) != 2:
            raise ValueError("Each interval needs a start and end in seconds.")
        start, end = map(float, interval)
        if not (np.isfinite(start) and np.isfinite(end) and 0 <= start < end <= info.duration):
            raise ValueError("Each interval must lie within the source duration.")
        # Quantize inward: a window must never read outside an approved interval.
        first = int(np.ceil(start * info.samplerate))
        last = info.frames if end == info.duration else int(np.floor(end * info.samplerate))
        if last <= first or round((last - first) * SAMPLE_RATE / info.samplerate) < 1:
            raise ValueError(
                "An interval is shorter than one audio sample at the source or target rate."
            )
        if mode in {"clips", "placement"} and last - first > native_size:
            raise ValueError(
                "Hard-negative clip exceeds the window; trim manually or enlarge the window."
            )
        if mode not in {"clips", "placement"} and last - first < native_size:
            raise ValueError("Every interval must contain at least one complete window.")
        ranges.append((first, last))
    ranges.sort()
    if any(current[0] < previous[1] for previous, current in zip(ranges, ranges[1:])):
        raise ValueError("Approved intervals must not overlap.")
    split = metadata.get("split", "unassigned") if split is None else split
    if split not in {"unassigned", "train", "validation", "test"}:
        raise ValueError("Unknown dataset split.")
    if "window_split" in metadata and metadata["window_split"] != split:
        raise ValueError(
            f"This source is locked to split {metadata['window_split']}; keep the same split."
        )
    seed = uuid4().int % (2**32) if seed is None else int(seed)
    if mode in {"random", "placement"}:
        if not isinstance(count, (int, np.integer)) or isinstance(count, bool) or count < 1:
            raise ValueError("Sample count must be a positive integer.")
    if mode == "random":
        lengths = [last - first - native_size + 1 for first, last in ranges]
        cumulative = np.cumsum(lengths).tolist()
        if count > cumulative[-1]:
            raise ValueError("Requested count exceeds the number of unique window positions.")
        # Sampling an integer range avoids allocating an array for an hours-long track.
        positions = random.Random(seed).sample(range(cumulative[-1]), count)
        selected = []
        for position in positions:
            index = bisect_right(cumulative, position)
            offset = position - (cumulative[index - 1] if index else 0)
            start = ranges[index][0] + offset
            selected.append((start, start + native_size))
    elif mode == "sliding":
        stride_seconds = window_seconds if stride_seconds is None else stride_seconds
        if not np.isfinite(stride_seconds) or stride_seconds <= 0:
            raise ValueError("Sliding stride must be positive and finite.")
        stride = round(stride_seconds * info.samplerate)
        if stride < 1:
            raise ValueError("Sliding stride must be at least one source sample.")
        selected = (
            (start, start + native_size)
            for first, last in ranges
            for start in range(first, last - native_size + 1, stride)
        )
    else:
        selected = iter(ranges)
    relative = str(source.relative_to(ROOT))
    folder = next_path(dataset_path(kind, "processed"), source.stem + "_extract", suffix="")
    source_group = metadata.get("source_group", relative)
    metadata.update(split=split, window_split=split, source_group=source_group)
    write_metadata(sidecar, metadata)
    outputs = []
    placement_rng = np.random.default_rng(seed) if mode == "placement" else None
    for start, end in selected:
        audio = read_normalized_segment(source, start, end - start)
        if not len(audio):
            raise ValueError("Selected clip is shorter than one target sample.")
        if mode in {"clips", "placement"} and len(audio) > window_size:
            raise ValueError(
                "The normalized clip exceeds the output window; no audio was truncated."
            )
        # Complete-window crops may differ by one sample due to resampling rounding.
        audio = audio[:window_size]
        for copy_index in range(count if mode == "placement" else 1):
            slack = window_size - len(audio)
            if mode == "placement":
                left = int(placement_rng.integers(0, slack + 1))
            else:
                left = slack // 2 if mode == "clips" else 0
            right = slack - left
            placed = np.pad(audio, (left, right))
            path = folder / f"window_{len(outputs) + 1:04d}.wav"
            write_wav(path, placed, overwrite=False)
            write_metadata(
                path.with_suffix(".json"),
                {
                    "source": relative,
                    "recording_id": source.stem,
                    "category": kind,
                    "label": 0,
                    "sample_rate": SAMPLE_RATE,
                    "speaker_id": metadata.get("speaker_id", SPEAKER_ID),
                    "split": split,
                    "source_group": source_group,
                    "selection_seconds": [start / info.samplerate, end / info.samplerate],
                    "source_start_frame": start,
                    "source_end_frame": end,
                    "source_sample_rate": info.samplerate,
                    "window_seconds": window_size / SAMPLE_RATE,
                    "windowing_mode": mode,
                    "sampling_seed": seed if mode in {"random", "placement"} else None,
                    "placement_offset_samples": left if mode == "placement" else None,
                    "placement_offset_seconds": left / SAMPLE_RATE if mode == "placement" else None,
                    "padding_left_samples": left,
                    "padding_right_samples": right,
                },
            )
            outputs.append(path)
    write_metadata(
        folder / "extraction.json",
        {
            "source": relative,
            "source_group": source_group,
            "category": kind,
            "label": 0,
            "split": split,
            "mode": mode,
            "window_seconds": window_size / SAMPLE_RATE,
            "approved_intervals_seconds": [
                [a / info.samplerate, b / info.samplerate] for a, b in ranges
            ],
            "seed": seed if mode in {"random", "placement"} else None,
            "copies_per_clip": count if mode == "placement" else None,
            "stride_seconds": stride / info.samplerate if mode == "sliding" else None,
            "window_count": len(outputs),
        },
    )
    log("Saved", len(outputs), kind, "windows:", folder.relative_to(ROOT))
    return outputs


def processed_negative_windows(kind, report_invalid=False):
    if kind not in {"negative", "hard_negative"}:
        raise ValueError("Choose a negative category.")
    outputs = []
    for path in sorted(dataset_path(kind, "processed").rglob("*.wav")):
        sidecar = path.with_suffix(".json")
        try:
            if not sidecar.exists():
                raise ValueError("Human negative samples need their extraction metadata.")
            metadata = json.loads(sidecar.read_text())
            if (
                not isinstance(metadata, dict)
                or metadata.get("category") != kind
                or metadata.get("label") != 0
            ):
                raise ValueError("Extraction metadata must match the category and label 0.")
        except (OSError, ValueError) as error:
            if report_invalid:
                log("Skipped human processed sample:", path.relative_to(DATA_DIR), "-", str(error))
            continue
        outputs.append(path)
    return outputs


def prepare_noise_library():
    # Load only the segment needed for mixing; long tracks are never normalized in full.
    return dataset_path("noise", "raw")


def available_audio_files(directory):
    candidates = []
    for path in sorted(directory.glob("*.wav")):
        sidecar = path.with_suffix(".json")
        try:
            metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
            if metadata.get("status", "complete") != "complete":
                continue
            if sf.info(path).frames > 0:
                candidates.append(path)
        except (OSError, ValueError, RuntimeError):
            continue
    return candidates


def make_noise_transform(noise_dir):
    candidates = available_audio_files(noise_dir)
    if candidates:
        return AddBackgroundNoise(sounds_path=candidates, min_snr_db=10, max_snr_db=25, p=1)
    return AddGaussianSNR(min_snr_db=10, max_snr_db=25, p=1)


def augmentation_sources(kind, group=None, report_invalid=False):
    if group not in {None, "human", "synthetic"}:
        raise ValueError("Choose human or synthetic augmentation sources.")
    human = []
    if group != "synthetic":
        human = (
            processed_positive_files()
            if kind == "positive"
            else processed_negative_windows(kind, report_invalid=report_invalid)
        )
    synthetic = []
    if group != "human" and kind in {"positive", "negative"}:
        synthetic = sorted(p for p in dataset_path(kind, "synthetic").glob("*.wav") if p.is_file())
    return human + synthetic


def source_group_label(kind, path):
    base = DATA_DIR / kind / "synthetic"
    if path.parent == base:
        return "synthetic/" + path.name
    return "human/" + str(path.relative_to(dataset_path(kind, "processed")))


def validate_positive_processed(path, window_seconds):
    info = sf.info(path)
    if info.samplerate != SAMPLE_RATE or info.channels != 1:
        raise ValueError("Prepared positive audio must be mono 16 kHz WAV.")
    if info.frames <= 0:
        raise ValueError("Prepared positive audio must be nonempty.")
    if not np.isfinite(window_seconds) or window_seconds <= 0:
        raise ValueError("Output window must be a positive finite duration.")
    if info.frames > round(window_seconds * SAMPLE_RATE):
        raise PositiveWindowError(
            "Prepared phrase exceeds the output window; choose a larger Window or prepare a shorter phrase."
        )


def augment_recording(kind, processed_source, window_seconds=2.0, show_players=False):
    if kind not in {"positive", "negative", "hard_negative"}:
        raise ValueError("Only positive, negative, and hard_negative examples are augmented here.")
    processed_source = Path(processed_source)
    synthetic = processed_source.parent == DATA_DIR / kind / "synthetic"
    if kind == "positive":
        validate_positive_processed(processed_source, window_seconds)
    audio, rate = sf.read(processed_source, dtype="float32")
    if rate != SAMPLE_RATE or audio.ndim != 1:
        raise ValueError("Process the source into mono 16 kHz audio first.")
    if not len(audio) or not np.isfinite(audio).all():
        raise ValueError("Processed audio must contain nonempty, finite samples.")
    sidecar = processed_source.with_suffix(".json")
    if sidecar.exists():
        metadata = json.loads(sidecar.read_text())
        if not isinstance(metadata, dict):
            raise ValueError("Processed metadata must be a JSON object.")
    elif kind == "positive" or synthetic:
        metadata = {
            "source": str(processed_source.relative_to(ROOT)),
            "source_stage": "processed",
            "recording_id": processed_source.stem,
            "speaker_id": "unassigned",
            "split": "unassigned",
        }
    else:
        raise ValueError("Saved negative samples require their extraction metadata.")
    label = 1 if kind == "positive" else 0
    if metadata.get("category", kind) != kind or metadata.get("label", label) != label:
        raise ValueError("Prepared metadata category and label must match its dataset folder.")
    if metadata.get("status", "complete") != "complete":
        raise ValueError("Only completed prepared samples can be augmented.")
    metadata.update(
        category=kind,
        label=label,
        sample_rate=SAMPLE_RATE,
        source_type="synthetic" if synthetic else "human",
    )
    seed = uuid4().int % (2**32)
    random.seed(seed)
    noise_dir = prepare_noise_library()
    if kind == "positive":
        variants = make_variants(audio, window_seconds, noise_dir, seed)
    else:
        # Negative windows already have fixed length; avoid three identical placement copies.
        quiet = Gain(min_gain_db=-9, max_gain_db=-3, p=1)
        stretch = TimeStretch(min_rate=1.03, max_rate=1.10, leave_length_unchanged=True, p=1)
        noise = make_noise_transform(noise_dir)
        variants = []
        for name, transform in [("quieter", quiet), ("noise", noise), ("faster", stretch)]:
            result = transform(samples=audio.copy(), sample_rate=SAMPLE_RATE)
            variants.append((name, result, dict(transform.parameters)))
    prefix = ("synthetic_" if synthetic else "") + processed_source.stem
    if (
        not synthetic
        and kind in {"negative", "hard_negative"}
        and processed_source.parent != dataset_path(kind, "processed")
    ):
        prefix = processed_source.parent.name + "_" + prefix
    folder = next_path(dataset_path(kind, "augmented"), prefix + "_run", suffix="")
    entries = []
    for name, result, parameters in variants:
        result = write_wav(folder / f"{name}.wav", result, overwrite=False)
        parameters = dict(parameters)
        if "noise_file_path" in parameters:
            parameters["noise_file_path"] = str(
                Path(parameters["noise_file_path"]).relative_to(ROOT)
            )
        entries.append({"file": f"{name}.wav", "parameters": parameters})
        if show_players:
            show_audio(result, name)
    write_metadata(
        folder / "metadata.json",
        {
            **metadata,
            "processed_source": str(processed_source.relative_to(ROOT)),
            "window_seconds": len(variants[0][1]) / SAMPLE_RATE,
            "augmentation_seed": seed,
            "variants": entries,
        },
    )
    log("Saved", len(variants), "variants:", folder.relative_to(ROOT))
    return folder


def import_legacy_recordings():
    # Copy old sources without deleting or re-encoding them. Repeated imports are safe.
    imported = []
    patterns = {
        "positive": "raw/positive_pilot/**/*.wav",
        "noise": "negatives/room_noise_pilot/**/*.wav",
    }
    for kind, pattern in patterns.items():
        known = {
            json.loads(path.read_text()).get("legacy_source")
            for path in dataset_path(kind, "raw").glob("*.json")
        }
        for source in sorted(DATA_DIR.glob(pattern)):
            relative = str(source.relative_to(ROOT))
            if relative in known:
                continue
            destination = next_path(dataset_path(kind, "raw"), kind)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with source.open("rb") as src, destination.open("xb") as dst:
                shutil.copyfileobj(src, dst)
            old_sidecar = source.with_suffix(".json")
            write_metadata(
                destination.with_suffix(".json"),
                {
                    "recording_id": destination.stem,
                    "category": kind,
                    "speaker_id": SPEAKER_ID,
                    "split": "unassigned",
                    "legacy_source": relative,
                    "raw_sample_rate": sf.info(destination).samplerate,
                    "original_metadata": json.loads(old_sidecar.read_text())
                    if old_sidecar.exists()
                    else {},
                },
            )
            process_recording(kind, destination)
            known.add(relative)
            imported.append(destination)
    return imported


def show_audio(audio, title, target=None):
    peak = float(np.max(np.abs(audio)))
    log(f"{title}: {len(audio) / SAMPLE_RATE:.2f}s, peak={peak:.3f}", target=target)
    playback = audio
    if peak > 1:
        # Float capture and resampling can exceed the player's [-1, 1] range.
        playback = audio * (0.999 / peak)
        log(
            "Preview volume reduced to fit playback range; source audio is unchanged.",
            target=target,
        )
    emit_preview(Audio(playback, rate=SAMPLE_RATE, normalize=False), target=target)
    # Display every bin's peak so short transients remain visible.
    bins = np.array_split(np.abs(audio), min(500, len(audio)))
    peaks = [float(b.max()) for b in bins]
    lines = "".join(
        f'<line x1="{i * 1.2:.1f}" x2="{i * 1.2:.1f}" '
        f'y1="{45 - v * 42:.1f}" y2="{45 + v * 42:.1f}" '
        'stroke="#2475c0"/>'
        for i, v in enumerate(peaks)
    )
    emit_preview(
        SVG(
            f'<svg xmlns="http://www.w3.org/2000/svg" width="600" height="90" '
            f'viewBox="0 0 600 90"><rect width="600" height="90" fill="#f3f6fa"/>{lines}</svg>'
        ),
        target=target,
    )


def random_place(phrase, window_seconds, rng):
    total = round(window_seconds * SAMPLE_RATE)
    if not 0 < len(phrase) <= total:
        raise ValueError(
            "The selected phrase does not fit. Adjust its bounds or increase the window."
        )
    offset = int(rng.integers(0, total - len(phrase) + 1))
    result = np.zeros(total, dtype=np.float32)
    result[offset : offset + len(phrase)] = phrase
    return result, offset / SAMPLE_RATE


def make_variants(phrase, window_seconds, noise_dir, seed):
    rng = np.random.default_rng(seed)
    if len(phrase) > round(window_seconds * SAMPLE_RATE):
        raise ValueError(
            "Phrase exceeds the window; no positive was truncated. Adjust selection or window."
        )
    variants = []
    for i in range(3):
        audio, offset = random_place(phrase, window_seconds, rng)
        variants.append((f"placement_{i + 1}", audio, {"offset_seconds": offset}))
    quiet = Gain(min_gain_db=-9, max_gain_db=-3, p=1)
    quieter = quiet(samples=phrase.copy(), sample_rate=SAMPLE_RATE)
    audio, offset = random_place(quieter, window_seconds, rng)
    variants.append(("quieter", audio, {"offset_seconds": offset, **quiet.parameters}))
    audio, offset = random_place(phrase, window_seconds, rng)
    noise = make_noise_transform(noise_dir)
    label = "room_noise" if isinstance(noise, AddBackgroundNoise) else "gaussian_noise_demo"
    noisy = noise(samples=audio.copy(), sample_rate=SAMPLE_RATE)
    variants.append((label, noisy, {"offset_seconds": offset, **noise.parameters}))
    # Faster speech fits whenever the original phrase fits. Preserve pitch and do not pad/crop it here.
    stretch = TimeStretch(min_rate=1.03, max_rate=1.10, leave_length_unchanged=False, p=1)
    faster = stretch(samples=phrase.copy(), sample_rate=SAMPLE_RATE)
    audio, offset = random_place(faster, window_seconds, rng)
    variants.append(("faster", audio, {"offset_seconds": offset, **stretch.parameters}))
    return variants


class AudioRecorder:
    """Capture native-rate blocks to disk while the notebook remains interactive."""

    def __init__(self):
        self.path = None
        self.kind = None
        self.frames = 0
        self.rate = 0
        self.error = None
        self.overflows = 0
        self.ready = Event()
        self._stop = Event()
        self._thread = None
        self._stream = None
        self._metadata = {}

    @property
    def running(self):
        return self._thread is not None and self._thread.is_alive()

    def start(self, device, kind="noise"):
        if self.running:
            raise RuntimeError("A recording is already running.")
        if kind not in {"positive", "negative", "hard_negative", "noise"}:
            raise ValueError("Unknown recording category.")
        info = sd.query_devices(device, "input")
        rate = int(info["default_samplerate"])
        sd.check_input_settings(device=device, channels=1, samplerate=rate, dtype="float32")
        if not sf.check_format("RF64", "FLOAT"):
            raise RuntimeError("This libsndfile installation does not support RF64 float WAV.")
        path = next_path(dataset_path(kind, "raw"), kind)
        path.parent.mkdir(parents=True, exist_ok=True)
        handle = path.open("xb")
        self.path, self.rate, self.kind = path, rate, kind
        self.frames, self.overflows, self.error = 0, 0, None
        self.ready.clear()
        self._stop.clear()
        self._metadata = {
            "recording_id": path.stem,
            "category": kind,
            "label": {"positive": 1, "negative": 0, "hard_negative": 0, "noise": None}[kind],
            "speaker_id": SPEAKER_ID,
            "split": "unassigned",
            "device": str(info["name"]),
            "raw_sample_rate": rate,
            "capture_rate": rate,
            "format": "RF64",
            "subtype": "FLOAT",
            "created_at": datetime.now(timezone.utc).isoformat(),
            "status": "recording",
        }
        try:
            write_metadata(path.with_suffix(".json"), self._metadata)
            self._thread = Thread(target=self._record, args=(handle, device), daemon=True)
            self._thread.start()
        except Exception:
            handle.close()
            raise
        return path

    def _record(self, handle, device):
        try:
            # RF64 keeps the .wav extension and can exceed the ordinary WAV 4 GiB limit.
            with handle:
                with sf.SoundFile(
                    handle,
                    mode="w",
                    samplerate=self.rate,
                    channels=1,
                    format="RF64",
                    subtype="FLOAT",
                ) as sink:
                    with sd.InputStream(
                        device=device,
                        samplerate=self.rate,
                        channels=1,
                        dtype="float32",
                        blocksize=4096,
                    ) as stream:
                        self._stream = stream
                        self.ready.set()
                        while not self._stop.is_set():
                            block, overflowed = stream.read(4096)
                            if len(block):
                                sink.write(block)
                                sink.flush()
                                self.frames += len(block)
                            self.overflows += int(overflowed)
        except Exception as error:
            self.error = error
        finally:
            self._stream = None
            self._metadata.update(
                status="failed" if self.error else ("complete" if self.frames else "empty"),
                frames=self.frames,
                duration_seconds=self.frames / self.rate,
                overflow_count=self.overflows,
                stopped_at=datetime.now(timezone.utc).isoformat(),
                error=str(self.error) if self.error else None,
            )
            try:
                write_metadata(self.path.with_suffix(".json"), self._metadata)
            except Exception as error:
                self.error = error
            self.ready.set()

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2)
            if self.running and self._stream is not None:
                self._stream.abort()
                self._thread.join(timeout=2)
            if self.running:
                raise RuntimeError(
                    "Microphone has not stopped yet. Keep the kernel open and retry Stop."
                )
        return self.path


def preview_recording(path, seconds=5):
    # Preview a bounded prefix rather than reading an hours-long recording into memory.
    info = sf.info(path)
    audio, rate = sf.read(
        path,
        frames=min(info.frames, round(seconds * info.samplerate)),
        dtype="float32",
        always_2d=True,
    )
    if not len(audio):
        return
    audio = audio.mean(axis=1)
    if rate != SAMPLE_RATE:
        factor = gcd(rate, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // factor, rate // factor)
    show_audio(
        audio.astype(np.float32), f"Recording preview: first {min(seconds, info.duration):.1f}s"
    )


previous_recorder = globals().get("_recorder") or globals().get("_noise_recorder")
if previous_recorder is not None and previous_recorder.running:
    raise RuntimeError("Stop recording before rerunning the control panel.")
previous_status_task = globals().get("_recording_status_task")
if previous_status_task is not None:
    previous_status_task.cancel()
previous_panel = globals().get("_pilot_panel")
if previous_panel is not None:
    close_panel(previous_panel)
_recorder = AudioRecorder()
_recording_status_task = None

input_options = [("System default microphone", None)] + [
    (f"{i}: {d['name']}", i)
    for i, d in enumerate(sd.query_devices())
    if d["max_input_channels"] > 0
]
device = widgets.Dropdown(
    options=input_options, description="Microphone:", layout={"width": "650px"}
)
window = widgets.Dropdown(options=[1.5, 2.0, 2.5, 3.0], value=2.5, description="Window (s):")
source_picker = widgets.Dropdown(description="Positive:", layout={"width": "650px"})
selection = widgets.FloatRangeSlider(
    value=(0, 1.0),
    min=0,
    max=1.0,
    step=0.01,
    description="Phrase (s):",
    readout_format=".2f",
    continuous_update=False,
    layout={"width": "650px"},
)
record_button = widgets.Button(description="Start positive", button_style="success")
negative_button = widgets.Button(description="Start negative")
hard_negative_button = widgets.Button(description="Start hard negative", layout={"width": "180px"})
hard_negative_stop_button = widgets.Button(
    description="Stop hard negative",
    button_style="danger",
    disabled=True,
    layout={"width": "180px"},
)
record_stop_button = widgets.Button(
    description="Stop positive", button_style="danger", disabled=True
)
negative_stop_button = widgets.Button(
    description="Stop negative", button_style="danger", disabled=True
)
noise_button = widgets.Button(description="Start noise", button_style="success")
noise_stop_button = widgets.Button(description="Stop noise", button_style="danger", disabled=True)
recording_status = widgets.HTML(value="Recorder idle. Click Start, then Stop when finished.")
preview_button = widgets.Button(description="Preview selection")
save_positive_button = widgets.Button(
    description="Save positive sample", button_style="info", layout={"width": "200px"}
)
positive_save_summary = widgets.HTML()
positive_aug_scope = widgets.Dropdown(
    options=[
        ("Selected processed files", "selected"),
        ("Augment all human", "human"),
        ("Augment all synthetic", "synthetic"),
    ],
    value="selected",
    description="Augment:",
    layout={"width": "550px"},
)
positive_processed = widgets.SelectMultiple(
    options=[],
    value=(),
    description="Files:",
    rows=7,
    layout={"width": "750px"},
)
positive_processed_refresh = widgets.Button(description="Refresh positive files")
positive_aug_button = widgets.Button(description="Augment selected", layout={"width": "230px"})
positive_aug_summary = widgets.HTML()
negative_aug_button = widgets.Button(description="Augment selected", layout={"width": "220px"})
negative_aug_scope = widgets.Dropdown(
    options=[
        ("Selected processed files", "selected"),
        ("Augment all human", "human"),
        ("Augment all synthetic", "synthetic"),
    ],
    value="selected",
    description="Augment:",
    layout={"width": "550px"},
)
negative_processed = widgets.SelectMultiple(
    options=[],
    value=(),
    description="Files:",
    rows=7,
    layout={"width": "750px"},
)
negative_processed_refresh = widgets.Button(description="Refresh file list")
negative_aug_summary = widgets.HTML()
negative_window = widgets.Dropdown(
    options=window.options, value=window.value, description="Output (s):"
)
negative_summary = widgets.HTML()
negative_multiple = widgets.Checkbox(value=False, description="Use several time intervals")
negative_split_info = widgets.HTML(value="Split is locked for this source after the first save.")
negative_kind = widgets.Dropdown(
    options=[("Negative", "negative"), ("Hard negative", "hard_negative")], description="Category:"
)
negative_source = widgets.Dropdown(description="Source:", layout={"width": "650px"})
negative_info = widgets.HTML(value="Choose a completed negative recording.")
negative_ranges = widgets.Textarea(
    description="Ranges (s):",
    placeholder="One start,end pair per line, e.g. 10,30",
    layout={"width": "650px", "height": "90px"},
)
negative_mode = widgets.Dropdown(
    options=[("Random crops from recording", "random"), ("Slide through selection", "sliding")],
    description="Method:",
    layout={"width": "450px"},
)
negative_count = widgets.BoundedIntText(value=20, min=1, max=10000, description="Samples:")
negative_stride = widgets.FloatText(value=2.0, description="Stride (s):")
negative_split = widgets.Dropdown(
    options=["unassigned", "train", "validation", "test"], description="Split:"
)
negative_selection = widgets.FloatRangeSlider(
    value=(0, 0),
    min=0,
    max=1,
    step=0.01,
    description="Selection (s):",
    readout_format=".2f",
    continuous_update=False,
    disabled=True,
    layout={"width": "650px"},
    style={"description_width": "100px"},
)
negative_preview_button = widgets.Button(description="Preview selection", layout={"width": "180px"})
negative_use_range_button = widgets.Button(
    description="Replace interval list", layout={"width": "180px"}
)
negative_add_range_button = widgets.Button(
    description="Add selection to list", layout={"width": "180px"}
)
extract_button = widgets.Button(
    description="Save samples", button_style="info", layout={"width": "200px"}
)
legacy_button = widgets.Button(description="Import old recordings")
output = widgets.Output()
positive_preview_output = widgets.Output()
buttons = [
    record_button,
    record_stop_button,
    negative_button,
    negative_stop_button,
    hard_negative_button,
    hard_negative_stop_button,
    noise_button,
    noise_stop_button,
    preview_button,
    save_positive_button,
    positive_processed_refresh,
    positive_aug_button,
    negative_aug_button,
    negative_processed_refresh,
    negative_preview_button,
    negative_use_range_button,
    negative_add_range_button,
    extract_button,
    legacy_button,
]


all_controls = [
    *buttons,
    device,
    window,
    source_picker,
    selection,
    positive_aug_scope,
    positive_processed,
    negative_kind,
    negative_source,
    negative_ranges,
    negative_mode,
    negative_count,
    negative_stride,
    negative_split,
    negative_selection,
    negative_window,
    negative_multiple,
    negative_aug_scope,
    negative_processed,
]
negative_picker_state = {"updating": False}
negative_ui_state = {"updating": False}


stop_buttons = {
    "positive": record_stop_button,
    "negative": negative_stop_button,
    "hard_negative": hard_negative_stop_button,
    "noise": noise_stop_button,
}


def update_controls():
    recording = _recorder.running
    for control in all_controls:
        control.disabled = recording
    for control in [
        negative_selection,
        negative_preview_button,
        negative_use_range_button,
        negative_add_range_button,
    ]:
        control.disabled = recording or negative_source.value is None
    update_negative_preparation()
    update_processed_augmentation()
    update_positive_controls()
    for kind, button in stop_buttons.items():
        button.disabled = not (recording and _recorder.kind == kind)


def run_action(action):
    for control in all_controls:
        control.disabled = True
    output.outputs = ()
    try:
        action()
    except Exception as error:
        log(f"{type(error).__name__}: {error}")
    finally:
        update_controls()


async def monitor_recording():
    while _recorder.running:
        if _recorder.ready.is_set():
            elapsed = _recorder.frames / _recorder.rate
            recording_status.value = (
                f"Recording {_recorder.kind}: {elapsed:.1f} seconds. "
                f"Click Stop {_recorder.kind} to finish."
            )
        await asyncio.sleep(0.25)
    update_controls()
    if _recorder.error:
        recording_status.value = "Recording failed; the partial file is excluded from processing."
        log("Recording error:", _recorder.error)


def start_recording(kind):
    global _recording_status_task
    if _recording_status_task is not None:
        _recording_status_task.cancel()
    path = _recorder.start(device.value, kind)
    recording_status.value = (
        f"Opening microphone for {kind}; no countdown. Click Stop {kind} to finish."
    )
    log("Recording to:", path.relative_to(ROOT))
    log("Click Stop", kind, "when finished. There is no preset duration.")
    if kind == "positive":
        log("Say the complete Hey GPT phrase once. No automatic trimming is applied.")
    elif kind == "hard_negative":
        log(
            "Record a similar-sounding phrase or known false trigger, without the complete wake phrase."
        )
    else:
        log("Avoid saying the complete wake phrase in negative or noise recordings.")
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        pass
    else:
        _recording_status_task = loop.create_task(monitor_recording())


def stop_recording():
    global _recording_status_task
    if _recording_status_task is not None:
        _recording_status_task.cancel()
    path = _recorder.stop()
    duration = _recorder.frames / _recorder.rate
    kind = _recorder.kind
    recording_status.value = f"{kind.title()} saved: {duration:.2f} seconds."
    log("Saved original:", path.relative_to(ROOT))
    if _recorder.error:
        recording_status.value = "Recording failed; the partial file is excluded from processing."
        log("Recording error:", _recorder.error)
        return
    if _recorder.frames == 0:
        log("No audio was captured; this empty recording is excluded from processing.")
        return
    if _recorder.overflows:
        log(
            "Warning: input overflow occurred",
            _recorder.overflows,
            "time(s); review the recording.",
        )
    if kind == "positive":
        refresh_sources(path)
        load_positive(path)
    elif kind in {"negative", "hard_negative"}:
        negative_kind.value = kind
        refresh_negative_sources(path)
        log("Raw saved. Drag Selection, preview it, then click Save samples.")
        log("Drag Selection and click Preview selection to listen to the chosen interval.")
    else:
        preview_recording(path)


def refresh_sources(selected=None):
    state["updating_picker"] = True
    try:
        sources = available_audio_files(dataset_path("positive", "raw"))
        source_picker.options = [("Select a recording", None)] + [(p.name, str(p)) for p in sources]
        if selected is not None:
            source_picker.value = str(selected)
    finally:
        state["updating_picker"] = False


def load_positive(source):
    positive_preview_output.outputs = ()
    state.update(audio=None, source=None)
    update_positive_controls()
    source = Path(source)
    audio = normalize_source(source)
    _, interval, _ = positive_phrase(source, audio)
    state.update(audio=audio, source=source)
    selection.value = (0, 0)
    selection.max = len(audio) / SAMPLE_RATE
    selection.value = interval
    log("Raw source:", source.relative_to(ROOT))
    log(
        "Preview only. Choose Phrase, listen, then click Save positive sample. No processed file is saved by opening a recording."
    )
    show_audio(audio, "Original timing, normalized for preview", target=positive_preview_output)
    log(
        "The full recording is selected unless you saved a manual selection. Adjust Phrase only if needed."
    )
    update_positive_controls()


def preview_positive_source():
    positive_preview_output.outputs = ()
    show_audio(selected_phrase(), "Selected phrase", target=positive_preview_output)


def selected_phrase():
    if state["audio"] is None:
        raise ValueError("Record or select a positive sample first.")
    start, end = selection.value
    phrase = state["audio"][round(start * SAMPLE_RATE) : round(end * SAMPLE_RATE)]
    if not len(phrase):
        raise ValueError("Select a nonempty phrase interval.")
    return phrase


def processed_positive_files():
    # The folder identifies the label; imported synthetic WAVs need no raw or JSON.
    return sorted(p for p in dataset_path("positive", "processed").glob("*.wav") if p.is_file())


def save_positive_sample():
    selected_phrase()  # Validate that a raw source and a nonempty interval are selected.
    processed = process_recording(
        "positive", state["source"], bounds=selection.value, window_seconds=window.value
    )[0]
    refresh_positive_processed(selected=[processed])
    log("Saved processed phrase:", processed.relative_to(ROOT))
    log("No augmentation was generated. Choose processed files below when ready.")


def refresh_positive_processed(selected=None):
    paths = augmentation_sources("positive")
    previous = positive_processed.value if selected is None else tuple(str(p) for p in selected)
    positive_processed.options = [(source_group_label("positive", p), str(p)) for p in paths]
    allowed = {str(p) for p in paths}
    positive_processed.value = tuple(value for value in previous if value in allowed)
    if selected is not None:
        positive_aug_scope.value = "selected"
    update_positive_controls()


def update_positive_controls(change=None):
    recording = _recorder.running
    has_source = state["source"] is not None and state["audio"] is not None
    selection.disabled = recording or not has_source
    preview_button.disabled = recording or not has_source
    valid = False
    if has_source:
        phrase = selected_phrase() if selection.value[1] > selection.value[0] else []
        valid = 0 < len(phrase) <= round(window.value * SAMPLE_RATE)
        positive_save_summary.value = (
            f"Selected {len(phrase) / SAMPLE_RATE:.3f}s; Window {window.value:.3f}s. "
            + (
                "Save writes the phrase to processed only."
                if valid
                else "Choose a nonempty phrase that fits Window before saving."
            )
        )
    else:
        positive_save_summary.value = (
            "Record or select a raw positive, preview it, then save its bounds."
        )
    save_positive_button.disabled = recording or not valid
    group = positive_aug_scope.value
    all_files = group != "selected"
    count = (
        len(augmentation_sources("positive", group)) if all_files else len(positive_processed.value)
    )
    positive_processed.disabled = recording or all_files
    positive_processed_refresh.disabled = recording
    positive_aug_button.disabled = recording or count == 0
    positive_aug_button.description = (
        f"Augment all {group} ({count})" if all_files else f"Augment {count} selected"
    )
    positive_aug_summary.value = (
        f"{count} prepared file(s); 6 variants per valid file. "
        "Raw recordings and unsaved Phrase edits are not used."
    )


def augment_positive_files(paths, show_players=False):
    if not paths:
        raise ValueError(
            "No processed positives selected. Save a human phrase or copy prepared WAVs into positive/synthetic, then Refresh."
        )
    count = 0
    for processed in paths:
        try:
            validate_positive_processed(processed, window.value)
            if show_players:
                # Read processed audio only, never the currently selected raw source.
                audio, rate = sf.read(processed, dtype="float32")
                # augment_recording validates the format before publishing variant previews.
                if rate == SAMPLE_RATE and audio.ndim == 1 and len(audio):
                    show_audio(audio, "Processed phrase")
            augment_recording(
                "positive", processed, window_seconds=window.value, show_players=show_players
            )
            count += 1
        except (ValueError, OSError, RuntimeError) as error:
            log("Skipped processed positive:", processed.name, "-", str(error))
    log("Augmented", count, "of", len(paths), "processed positive file(s); raw was not processed.")
    return count


def augment_chosen_positives():
    if positive_aug_scope.value != "selected":
        chosen = augmentation_sources("positive", positive_aug_scope.value)
    else:
        available = {str(p): p for p in augmentation_sources("positive")}
        selected = positive_processed.value
        if any(value not in available for value in selected):
            raise ValueError(
                "A selected processed file is no longer available. Refresh the file list and select again."
            )
        chosen = [available[value] for value in selected]
    return augment_positive_files(chosen, show_players=positive_aug_scope.value == "selected")


def augment_all(kind, group="human"):
    if kind == "positive":
        return augment_positive_files(augmentation_sources(kind, group))
    outputs = augmentation_sources(kind, group)
    if not outputs:
        raise ValueError("No saved samples found. Select a source and click Save samples first.")
    for processed in outputs:
        augment_recording(kind, processed)
    log("Augmented", len(outputs), "existing", kind, "windows; no raw recording was reprocessed.")


def refresh_processed_samples(selected=None):
    paths = augmentation_sources(negative_kind.value, report_invalid=True)
    previous = negative_processed.value if selected is None else tuple(str(p) for p in selected)
    negative_processed.options = [
        (source_group_label(negative_kind.value, p), str(p)) for p in paths
    ]
    allowed = {str(p) for p in paths}
    negative_processed.value = tuple(value for value in previous if value in allowed)
    if selected is not None:
        negative_aug_scope.value = "selected"
    update_processed_augmentation()


def update_processed_augmentation(change=None):
    recording = _recorder.running
    available = len(negative_processed.options)
    scope = negative_aug_scope.value
    all_files = scope != "selected"
    count = (
        len(augmentation_sources(negative_kind.value, scope))
        if all_files
        else len(negative_processed.value)
    )
    negative_processed.disabled = recording or all_files
    negative_processed_refresh.disabled = recording
    negative_aug_button.disabled = recording or count == 0
    negative_aug_button.description = (
        f"Augment all {scope} ({count})" if all_files else f"Augment {count} selected"
    )
    negative_aug_summary.value = (
        f"{count} targeted of {available} prepared files in {negative_kind.value}. "
        "Human uses processed/; synthetic uses synthetic/. Each click creates new runs."
    )


def augment_chosen_negatives():
    kind = negative_kind.value
    if negative_aug_scope.value != "selected":
        chosen = augmentation_sources(kind, negative_aug_scope.value)
    else:
        available = {str(p): p for p in augmentation_sources(kind)}
        selected = negative_processed.value
        if any(value not in available for value in selected):
            raise ValueError(
                "A selected file is no longer available. Refresh the file list and select again."
            )
        chosen = [available[value] for value in selected]
    if not chosen:
        raise ValueError(
            "Select at least one prepared file, or choose Augment all human / synthetic."
        )
    count = 0
    for path in chosen:
        try:
            augment_recording(kind, path)
            count += 1
        except (ValueError, OSError, RuntimeError) as error:
            log("Skipped prepared negative:", path.name, "-", str(error))
    log("Augmented", count, "of", len(chosen), kind, "prepared file(s).")


def refresh_negative_sources(selected=None):
    negative_picker_state["updating"] = True
    try:
        sources = available_audio_files(dataset_path(negative_kind.value, "raw"))
        negative_source.options = [("Select a recording", None)] + [
            (p.name, str(p)) for p in sources
        ]
        if selected is not None:
            negative_source.value = str(selected)
    finally:
        negative_picker_state["updating"] = False
    load_negative_source()


def load_negative_source():
    negative_ui_state["updating"] = True
    negative_ranges.value = ""
    negative_multiple.value = False
    negative_selection.value = (0, 0)
    negative_selection.max = 1
    if negative_source.value is None:
        negative_info.value = "Choose a completed negative recording."
        negative_split.value = "unassigned"
        negative_ui_state["updating"] = False
        update_controls()
        return
    source = Path(negative_source.value)
    info = sf.info(source)
    sidecar = source.with_suffix(".json")
    metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
    negative_split.value = metadata.get("split", "unassigned")
    negative_info.value = (
        f"Recording: {info.duration:.2f} seconds. Drag the handles to choose a section."
    )
    negative_selection.max = info.duration
    negative_selection.value = (0, info.duration)
    negative_mode.value = "placement" if negative_kind.value == "hard_negative" else "random"
    negative_count.max = 10000
    negative_count.value = 1 if negative_kind.value == "hard_negative" else 20
    negative_ui_state["updating"] = False
    log("Selected", source.name, f"({info.duration:.3f}s).")
    update_controls()


def change_negative_category(change):
    negative_aug_scope.options = [
        ("Selected prepared files", "selected"),
        ("Augment all human", "human"),
    ] + ([("Augment all synthetic", "synthetic")] if change["new"] == "negative" else [])
    negative_aug_scope.value = "selected"
    negative_mode.options = (
        [
            ("Random position in output window", "placement"),
            ("Centered in output window", "clips"),
        ]
        if change["new"] == "hard_negative"
        else [("Random crops from recording", "random"), ("Slide through selection", "sliding")]
    )
    negative_mode.value = "placement" if change["new"] == "hard_negative" else "random"
    refresh_negative_sources()
    refresh_processed_samples(selected=[])


def parse_negative_ranges():
    intervals = []
    for line in negative_ranges.value.splitlines():
        if not line.strip():
            continue
        try:
            start, end = map(float, line.split(","))
        except ValueError as error:
            raise ValueError(
                "Use one start,end pair in seconds per line, for example 10,30."
            ) from error
        intervals.append((start, end))
    return intervals


def negative_extraction_intervals():
    if negative_multiple.value:
        return parse_negative_ranges()
    path, info, start, end, first, last = selected_negative_range()
    return [(start, end)]


def extract_selected_negatives():
    if negative_source.value is None:
        raise ValueError("Select a negative source first.")
    saved = extract_negative_windows(
        negative_kind.value,
        negative_source.value,
        negative_extraction_intervals(),
        window_seconds=window.value,
        mode=negative_mode.value,
        count=negative_count.value,
        stride_seconds=negative_stride.value,
        split=negative_split.value,
    )

    refresh_processed_samples(selected=saved)


def update_negative_preparation(change=None):
    if negative_ui_state["updating"]:
        return
    negative_ui_state["updating"] = True
    try:
        recording = _recorder.running
        method = negative_mode.value
        negative_count.layout.display = "" if method in {"random", "placement"} else "none"
        negative_count.description = "Copies / clip:" if method == "placement" else "Samples:"
        negative_count.style.description_width = "100px" if method == "placement" else "initial"
        if method == "placement":
            negative_count.max = 10000
        negative_stride.layout.display = "" if method == "sliding" else "none"
        negative_ranges_box.layout.display = "" if negative_multiple.value else "none"
        for control in [negative_ranges, negative_use_range_button, negative_add_range_button]:
            control.disabled = (
                recording or negative_source.value is None or not negative_multiple.value
            )
        try:
            if negative_source.value is None:
                raise ValueError("Choose a recording above to prepare samples.")
            info = sf.info(negative_source.value)
            sidecar = Path(negative_source.value).with_suffix(".json")
            source_metadata = json.loads(sidecar.read_text()) if sidecar.exists() else {}
            locked_split = source_metadata.get("window_split")
            if locked_split is not None:
                negative_split.value = locked_split
                negative_split.disabled = True
                negative_split_info.value = (
                    f"This recording is locked to {locked_split}; all its samples use that split."
                )
            else:
                negative_split.disabled = recording
                negative_split_info.value = "Choose a split before the first save; it will then be locked for this recording."
            intervals = negative_extraction_intervals()
            if not intervals:
                raise ValueError(
                    "Add at least one interval in Advanced, or turn off several intervals to use the slider."
                )
            frames = []
            for start, end in sorted(intervals):
                if not (
                    np.isfinite(start) and np.isfinite(end) and 0 <= start < end <= info.duration
                ):
                    raise ValueError("Choose a nonempty section inside the recording.")
                first = int(np.ceil(start * info.samplerate))
                last = info.frames if end == info.duration else int(np.floor(end * info.samplerate))
                if last <= first or round((last - first) * SAMPLE_RATE / info.samplerate) < 1:
                    raise ValueError("Choose a section containing at least one audio sample.")
                if frames and first < frames[-1][1]:
                    raise ValueError("Time intervals overlap. Remove the overlap in Advanced.")
                frames.append((first, last))
            target = round(round(window.value * SAMPLE_RATE) * info.samplerate / SAMPLE_RATE)
            sizes = [b - a for a, b in frames]
            duration = sum(sizes) / info.samplerate
            prefix = f"Selected: {duration:.2f}s. Each saved sample: {window.value:.2f}s. "
            if method in {"clips", "placement"}:
                if any(size > target for size in sizes):
                    raise ValueError(prefix + "Select a shorter clip or increase Output (s).")
                padding = sum(max(0, target - size) for size in sizes) / info.samplerate
                if method == "placement":
                    total = len(sizes) * negative_count.value
                    message = (
                        prefix
                        + f"Keep each complete clip. Save {negative_count.value} random placement(s) per clip ({total} total). Fill the rest with silence."
                    )
                    if all(size == target for size in sizes):
                        message += " The clip fills the output window, so its position cannot move."
                else:
                    total = len(sizes)
                    message = (
                        prefix
                        + f"Save {total} centered sample(s); add {padding:.2f}s of silence in total."
                    )
                extract_button.description = (
                    "Save 1 sample" if total == 1 else f"Save {total} samples"
                )
            else:
                if any(size < target for size in sizes):
                    raise ValueError(prefix + "Select a longer section or reduce Output (s).")
                if method == "random":
                    capacity = sum(size - target + 1 for size in sizes)
                    negative_count.max = min(10000, capacity)
                    message = (
                        prefix
                        + f"Save {negative_count.value} random sample(s) inside the selected section(s). Overlap is allowed."
                    )
                else:
                    stride = negative_stride.value
                    if (
                        not np.isfinite(stride)
                        or stride <= 0
                        or round(stride * info.samplerate) < 1
                    ):
                        raise ValueError("Set a positive Stride in seconds.")
                    total = sum(
                        (size - target) // round(stride * info.samplerate) + 1 for size in sizes
                    )
                    message = prefix + f"Save {total} sample(s), moving {stride:.2f}s each time."
                extract_button.description = "Save samples"
            negative_summary.value = message
            extract_button.disabled = recording
        except ValueError as error:
            negative_summary.value = str(error)
            extract_button.disabled = True
    finally:
        negative_ui_state["updating"] = False


def toggle_multiple_intervals(change):
    if negative_ui_state["updating"]:
        return
    if change["new"] and not negative_ranges.value.strip() and negative_source.value is not None:
        start, end = negative_selection.value
        negative_ranges.value = f"{start!r},{end!r}"
    update_negative_preparation()


def sync_output_window(change):
    other = negative_window if change["owner"] is window else window
    if other.value != change["new"]:
        other.value = change["new"]
    update_negative_preparation()


def selected_negative_range():
    if negative_source.value is None:
        raise ValueError("Select a negative source first.")
    path = Path(negative_source.value)
    info = sf.info(path)
    start, end = negative_selection.value
    if not (np.isfinite(start) and np.isfinite(end) and 0 <= start < end <= info.duration):
        raise ValueError("Select a nonempty interval inside the recording.")
    first = int(np.ceil(start * info.samplerate))
    last = info.frames if end == info.duration else int(np.floor(end * info.samplerate))
    if last <= first or round((last - first) * SAMPLE_RATE / info.samplerate) < 1:
        raise ValueError("Select an interval containing at least one audio sample.")
    return path, info, start, end, first, last


def preview_negative_source():
    path, info, start, end, first, last = selected_negative_range()
    audio = read_normalized_segment(path, first, last - first)
    show_audio(audio, f"Selected source interval: {start:.3f}-{end:.3f}s")


def use_negative_range(append=False):
    path, info, start, end, first, last = selected_negative_range()
    intervals = parse_negative_ranges() if append else []
    intervals.append((start, end))
    intervals.sort()
    if any(
        not (np.isfinite(a) and np.isfinite(b) and 0 <= a < b <= info.duration)
        for a, b in intervals
    ):
        raise ValueError("Every range must be a nonempty interval inside this source.")
    if any(current[0] < previous[1] for previous, current in zip(intervals, intervals[1:])):
        raise ValueError(
            "Ranges overlap. Use range to replace the list, or select a separate interval."
        )
    negative_ranges.value = "\n".join(f"{a!r},{b!r}" for a, b in intervals)
    negative_multiple.value = True
    log("Interval list updated:", len(intervals), "interval(s).")
    update_negative_preparation()


def import_old():
    imported = import_legacy_recordings()
    refresh_sources(state["source"])
    log("Imported", len(imported), "recordings. Old files were preserved.")


def on_source_change(change):
    if state["updating_picker"]:
        return
    if change["new"] is None:
        positive_preview_output.outputs = ()
        state.update(audio=None, source=None)
        update_positive_controls()
    else:
        run_action(lambda: load_positive(change["new"]))


def action_row(controls, explanation):
    return widgets.HBox(
        [
            *controls,
            widgets.HTML(
                value=f"<span style='color:var(--vscode-descriptionForeground, #666)'>{explanation}</span>",
                layout={"flex": "1 1 320px"},
            ),
        ],
        layout={"align_items": "center", "flex_flow": "row wrap"},
    )


negative_ranges_box = widgets.VBox(
    [
        widgets.HTML(
            value="With several intervals enabled, this list replaces the slider for saving samples."
        ),
        negative_ranges,
        action_row(
            [negative_use_range_button], "Replace the interval list with the slider selection."
        ),
        action_row(
            [negative_add_range_button], "Append the slider selection to the interval list."
        ),
    ]
)
negative_advanced = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                negative_split,
                negative_split_info,
                negative_multiple,
                negative_ranges_box,
            ]
        )
    ],
    selected_index=None,
)
negative_advanced.set_title(0, "Advanced: dataset split and several intervals")
legacy_tools = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                action_row(
                    [legacy_button],
                    "Copy recordings from the old pilot folders into the current layout.",
                )
            ]
        )
    ],
    selected_index=None,
)
legacy_tools.set_title(0, "Legacy data tools")


refresh_sources()
refresh_positive_processed()
source_picker.observe(on_source_change, names="value")
refresh_negative_sources()
refresh_processed_samples()
negative_kind.observe(change_negative_category, names="value")
negative_source.observe(
    lambda change: (
        run_action(load_negative_source) if not negative_picker_state["updating"] else None
    ),
    names="value",
)
for control in [
    negative_selection,
    negative_mode,
    negative_count,
    negative_stride,
    negative_ranges,
    negative_split,
]:
    control.observe(update_negative_preparation, names="value")
negative_multiple.observe(toggle_multiple_intervals, names="value")
window.observe(sync_output_window, names="value")
negative_window.observe(sync_output_window, names="value")
extract_button.on_click(lambda _: run_action(extract_selected_negatives))
negative_preview_button.on_click(lambda _: run_action(preview_negative_source))
negative_use_range_button.on_click(lambda _: run_action(use_negative_range))
negative_add_range_button.on_click(lambda _: run_action(lambda: use_negative_range(append=True)))
hard_negative_button.on_click(lambda _: run_action(lambda: start_recording("hard_negative")))
hard_negative_stop_button.on_click(lambda _: run_action(stop_recording))
record_button.on_click(lambda _: run_action(lambda: start_recording("positive")))
record_stop_button.on_click(lambda _: run_action(stop_recording))
negative_button.on_click(lambda _: run_action(lambda: start_recording("negative")))
negative_stop_button.on_click(lambda _: run_action(stop_recording))
noise_button.on_click(lambda _: run_action(lambda: start_recording("noise")))
noise_stop_button.on_click(lambda _: run_action(stop_recording))
preview_button.on_click(lambda _: run_action(preview_positive_source))
save_positive_button.on_click(lambda _: run_action(save_positive_sample))
positive_processed_refresh.on_click(lambda _: run_action(refresh_positive_processed))
positive_aug_button.on_click(lambda _: run_action(augment_chosen_positives))
positive_processed.observe(update_positive_controls, names="value")
positive_aug_scope.observe(update_positive_controls, names="value")
selection.observe(update_positive_controls, names="value")
window.observe(update_positive_controls, names="value")
negative_aug_button.on_click(lambda _: run_action(augment_chosen_negatives))
negative_processed_refresh.on_click(lambda _: run_action(refresh_processed_samples))
negative_processed.observe(update_processed_augmentation, names="value")
negative_aug_scope.observe(update_processed_augmentation, names="value")
legacy_button.on_click(lambda _: run_action(import_old))
_pilot_panel = widgets.VBox(
    [
        device,
        window,
        source_picker,
        selection,
        action_row(
            [record_button, record_stop_button],
            "Record a positive example. Stop saves raw only; preview and Save positive sample below.",
        ),
        action_row(
            [negative_button, negative_stop_button],
            "Record ordinary negatives. Stop saves raw; use Save samples below to process it.",
        ),
        action_row(
            [hard_negative_button, hard_negative_stop_button],
            "Record a hard-negative example. Stop saves raw; use Save samples below to process it.",
        ),
        action_row(
            [noise_button, noise_stop_button],
            "Record background noise into noise/raw/ for mixing into augmented samples.",
        ),
        recording_status,
        action_row([preview_button], "Listen to the selected positive phrase."),
        positive_preview_output,
        action_row(
            [save_positive_button],
            "Save the reviewed Phrase to positive/processed only; no augmentation.",
        ),
        positive_save_summary,
        widgets.HTML(value="<b>Augment positive processed audio</b>"),
        widgets.HTML(
            value="Both teams use this list. Copy prepared synthetic WAVs directly into data/positive/synthetic/: mono 16 kHz, nonempty, phrase no longer than Window. JSON is optional. Use distinct names such as human_alex_001.wav or tts_voice01_001.wav."
        ),
        positive_aug_scope,
        positive_processed,
        action_row(
            [positive_processed_refresh],
            "Reload copied files. Ctrl/Cmd-click to select several files; Shift-click for a range. Save positive sample selects its output automatically.",
        ),
        widgets.HBox(
            [positive_aug_button, positive_aug_summary],
            layout={"align_items": "center", "flex_flow": "row wrap"},
        ),
        widgets.HTML(value="<b>Prepare negative samples</b>"),
        negative_kind,
        negative_source,
        negative_info,
        negative_selection,
        action_row(
            [negative_preview_button],
            "Listen to the current slider selection. Save samples creates the processed audio.",
        ),
        negative_window,
        negative_mode,
        negative_count,
        negative_stride,
        negative_summary,
        negative_advanced,
        action_row(
            [extract_button],
            "Selected Source + slider interval (or the enabled Advanced interval list) → processed/. Each click saves a new set.",
        ),
        widgets.HTML(value="<b>Augment processed audio</b>"),
        negative_aug_scope,
        negative_processed,
        action_row(
            [negative_processed_refresh],
            "Human inputs use processed/; prepared synthetic negative WAVs use negative/synthetic/. File labels identify the group and source. Ctrl/Cmd-click for several files; Shift-click for a range.",
        ),
        widgets.HBox(
            [negative_aug_button, negative_aug_summary],
            layout={"align_items": "center", "flex_flow": "row wrap"},
        ),
        legacy_tools,
        output,
    ]
)
display(_pilot_panel)
update_controls()
log("Ready. Recording starts only when you click a recording button.")
log("Noise is shared across sessions from data/noise/raw/.")